In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import numpy as mp
import matplotlib.dates as mdates

# Task 2: Data Aquisition & Initial Visualization

In [ ]:
#Load data

#please note that this is the only path I could get to work on my Mac! Please change your path as needed
duchesne = pd.read_csv("/uufs/chpc.utah.edu/common/home/u1600917/Hydroinformatics/Homework1/data/09277500_1980_2020.csv")
duchesne = duchesne.dropna(subset=["USGS_flow"]).drop(columns=['variable','USGS_ID','measurement_unit','qualifiers','series'])
duchesne["Datetime"] = pd.to_datetime(duchesne["Datetime"])


moon_reservoir = pd.read_csv("/uufs/chpc.utah.edu/common/home/u1600917/Hydroinformatics/Homework1/data/09291000_1980_2020.csv")
moon_reservoir = moon_reservoir.dropna(subset=["USGS_flow"]).drop(columns=['variable','USGS_ID','measurement_unit','qualifiers','series'])
moon_reservoir["Datetime"] = pd.to_datetime(moon_reservoir["Datetime"])

weber_headwater = pd.read_csv("/uufs/chpc.utah.edu/common/home/u1600917/Hydroinformatics/Homework1/data/10128500_1980_2020.csv")
weber_headwater = weber_headwater.dropna(subset=["USGS_flow"]).drop(columns=['variable','USGS_ID','measurement_unit','qualifiers','series'])
weber_headwater["Datetime"] = pd.to_datetime(weber_headwater["Datetime"])

provo = pd.read_csv("/uufs/chpc.utah.edu/common/home/u1600917/Hydroinformatics/Homework1/data/10154200_1980_2020.csv")
provo = provo.dropna(subset=["USGS_flow"]).drop(columns=['variable','USGS_ID','measurement_unit','qualifiers','series'])
provo["Datetime"] = pd.to_datetime(provo["Datetime"])


In [ ]:
#process the data, I chose to perform 2014,2015,2016,2017,2018, and 2019 as my 6 years
#note that I have gone back to convert my data into MGD here, as the later tasks were hard to analyze in cfs

duchesne = duchesne.loc[duchesne.index[duchesne["Datetime"].dt.year == 2014].min():duchesne.index[duchesne["Datetime"].dt.year == 2019].max()]
duchesne["USGS_flow"] = duchesne["USGS_flow"]*0.646

moon_reservoir = moon_reservoir.loc[moon_reservoir.index[moon_reservoir["Datetime"].dt.year == 2014].min():moon_reservoir.index[moon_reservoir["Datetime"].dt.year == 2019].max()]
moon_reservoir["USGS_flow"] = moon_reservoir["USGS_flow"]*0.646

weber_headwater = weber_headwater.loc[weber_headwater.index[weber_headwater["Datetime"].dt.year == 2014].min():weber_headwater.index[weber_headwater["Datetime"].dt.year == 2019].max()]
weber_headwater["USGS_flow"] = weber_headwater["USGS_flow"]*0.646

provo = provo.loc[provo.index[provo["Datetime"].dt.year == 2014].min():provo.index[provo["Datetime"].dt.year == 2019].max()]
provo["USGS_flow"] = provo["USGS_flow"]*0.646

streamflow = duchesne.merge(moon_reservoir.merge(weber_headwater.merge(provo,how="outer",on="Datetime",sort=True,suffixes = ['_weber','_provo']),how="outer",on="Datetime",sort=True),how="outer",on="Datetime",sort=True,suffixes=['_duch','_moon'])
streamflow.set_index("Datetime",inplace=True) #Note that this set_index requires to hit "run all" or the code doesn't work

#Rename columns to be accurate with data
streamflow = streamflow.rename(columns={"USGS_flow_duch":"Duchesne Site","USGS_flow_moon":"Moon Lake Reservoir", "USGS_flow_provo":"Provo Site","USGS_flow_weber":"Weber Headwater Catchment"})

In [ ]:
#Exploratory Data Analysis
streamflow.plot(title="Initial Timeseries Streamflow Data",
                figsize = (15,6),
                subplots=True,
                layout=(2,2),
                sharex=False,
                ylabel="Streamflow (MGD)")

# Task 3: Temporal Resampling & Statistical Aggregation

In [ ]:
#raw daily
daily = streamflow
#weekly mean
weekly = streamflow.resample('W').mean()
#monthly total
monthly = streamflow.resample('ME').sum()

In [ ]:
#Visualization
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(17,10))

plotA = ax[0,0]
plotB = ax[0,1]
plotC = ax[1,0]
fig.delaxes(ax[1,1])

#daily plot
plotA.plot(daily)
plotA.set_title("Daily (Raw)",fontsize = 20,y=1.03)
plotA.set_xlabel("Date",fontsize = 16,labelpad=10)
plotA.set_ylabel("Streamflow (MGD)", fontsize = 16,labelpad=10)
plotA.set_xlim(daily.index.min(),daily.index.max())
plotA.tick_params(axis='both', labelsize=12)


#weekly plot
plotB.plot(weekly)
plotB.set_title("Weekly Mean",fontsize = 20,y=1.03)
plotB.set_xlabel("Date",fontsize = 16,labelpad=10)
plotB.set_ylabel("Streamflow (MGD)", fontsize = 16,labelpad=10)
plotB.set_xlim(weekly.index.min(),weekly.index.max())
plotB.tick_params(axis='both', labelsize=12)

#monthly plot
plotC.plot(monthly)
plotC.set_title("Monthly Volume",fontsize = 20,y=1.03)
plotC.set_xlabel("Date",fontsize = 16,labelpad=10)
plotC.set_ylabel("Streamflow (MGD)", fontsize = 16,labelpad=10)
plotC.set_xlim(monthly.index.min(),monthly.index.max())
plotC.tick_params(axis='both', labelsize=12)

plotD.set_axis_off()

#Panel D -> legend
plt.subplots_adjust(hspace=0.5, wspace=0.3)
fig.legend(daily.columns.values, 
           loc='lower right',
           bbox_to_anchor=(0.87, 0.12),
           fontsize = 'xx-large',
           title = "Site of Data",
           labelspacing = 1,
           title_fontproperties = {'weight':'bold','size':24}
           )
plt.show()

# Task 4: Comparitive Analysis

In [ ]:
#locations are Moon Lake Reservoir (gauge is below it) and Weber Headwater Catchment(see source/writeup for proof)
#wet year = 2019, dry year = 2015 (note that moon lake is slightly drier in 2018, but headwater is significantly wetter, making 2015 a better choice)

#Restrict locations for all plot types
daily = daily.drop(columns=["Duchesne Site","Provo Site"])
daily = daily.reset_index(drop=False)
weekly = weekly.drop(columns=["Duchesne Site","Provo Site"])
weekly = weekly.reset_index(drop=False)
monthly = monthly.drop(columns=["Duchesne Site","Provo Site"])
monthly = monthly.reset_index(drop=False)

In [ ]:
#New cell block because the .drop() requires the "runall" button (too slow)
#Restrict time for plot types (water year is wet: Oct 2018- sep 2019, dry:Oct 2014- sep 2015)
wet_daily = daily.loc[daily.index[daily["Datetime"] == pd.Timestamp('2014-10-01')].min():daily.index[daily["Datetime"] == pd.Timestamp('2015-09-30')].min()].copy()
wet_daily["Datetime"] = pd.to_datetime(wet_daily["Datetime"])
wet_weekly = wet_daily.resample('W',on='Datetime').mean()
wet_monthly = wet_daily.resample('ME', on='Datetime').sum()

dry_daily = daily.loc[daily.index[daily["Datetime"] == pd.Timestamp('2018-10-01')].min():daily.index[daily["Datetime"] == pd.Timestamp('2019-09-30')].min()].copy()
dry_daily["Datetime"] = pd.to_datetime(dry_daily["Datetime"])
dry_weekly = dry_daily.resample('W',on='Datetime').mean()
dry_monthly = dry_daily.resample('ME', on='Datetime').sum()

#Show range/basic info of flow for each
moon_day = wet_daily["Moon Lake Reservoir"].describe()
weber_day = wet_daily['Weber Headwater Catchment'].describe()
print(moon_day, weber_day, sep='\n')

#get aggregate for the wet year using daily values
wet_daily["Moon Lake Reservoir"] = wet_daily["Moon Lake Reservoir"].cumsum()
wet_daily["Weber Headwater Catchment"] = wet_daily["Weber Headwater Catchment"].cumsum()
#get aggregate for the wet year using weekly average values
wet_weekly["Moon Lake Reservoir"] = wet_weekly["Moon Lake Reservoir"].cumsum()
wet_weekly["Weber Headwater Catchment"] = wet_weekly["Weber Headwater Catchment"].cumsum()

#get aggregate for the dry year using daily values
dry_daily["Moon Lake Reservoir"] = dry_daily["Moon Lake Reservoir"].cumsum()
dry_daily["Weber Headwater Catchment"] = dry_daily["Weber Headwater Catchment"].cumsum()
#get aggregate for the dry year using weekly average values
dry_weekly["Moon Lake Reservoir"] = dry_weekly["Moon Lake Reservoir"].cumsum()
dry_weekly["Weber Headwater Catchment"] = dry_weekly["Weber Headwater Catchment"].cumsum()


In [ ]:
#fix data for plots
wet_daily["Datetime"] = pd.to_datetime(wet_daily["Datetime"])
wet_daily.set_index("Datetime", inplace=True)

# wet_weekly["Datetime"] = pd.to_datetime(wet_weekly["Datetime"])
# wet_weekly.set_index("Datetime", inplace=True)

dry_daily["Datetime"] = pd.to_datetime(dry_daily["Datetime"])
dry_daily.set_index("Datetime", inplace=True)

# dry_weekly["Datetime"] = pd.to_datetime(dry_weekly["Datetime"])
# dry_weekly.set_index("Datetime", inplace=True)

In [ ]:
#plots 

myFmt = mdates.DateFormatter("%y-%m") 

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(20,10))

plot1 = ax[0,0]
plot2 = ax[0,1]
plot3 = ax[1,0]
plot4 = ax[1,1]

plot1.plot(wet_daily)
plot1.set_title("Aggregate Daily Flow (Wet Year)",fontsize = 20,y=1.03)
plot1.set_xlabel("Date",fontsize = 14,labelpad=10)
plot1.set_ylabel("Streamflow (MGD)", fontsize = 16,labelpad=10)
plot1.set_xlim(wet_daily.index.min(),wet_daily.index.max())
plot1.tick_params(axis='both', labelsize=12)
plot1.xaxis.set_major_formatter(myFmt)

plot2.plot(dry_daily)
plot2.set_title("Aggregate Daily Flow (Dry Year)",fontsize = 20,y=1.03)
plot2.set_xlabel("Date",fontsize = 14,labelpad=10)
plot2.set_ylabel("Streamflow (MGD)", fontsize = 16,labelpad=10)
plot2.set_xlim(dry_daily.index.min(),dry_daily.index.max())
plot2.tick_params(axis='both', labelsize=12)
plot2.xaxis.set_major_formatter(myFmt)

plot3.plot(wet_weekly)
plot3.set_title("Aggregate Average Weekly Flow (Wet Year)",fontsize = 20,y=1.03)
plot3.set_xlabel("Date",fontsize = 14,labelpad=10)
plot3.set_ylabel("Streamflow (MGD)", fontsize = 16,labelpad=10)
plot3.set_xlim(wet_weekly.index.min(),wet_weekly.index.max())
plot3.tick_params(axis='both', labelsize=12)
plot3.xaxis.set_major_formatter(myFmt)

plot4.plot(dry_weekly)
plot4.set_title("Aggregate Average Weekly Flow (Dry Year)",fontsize = 20,y=1.03)
plot4.set_xlabel("Date",fontsize = 14,labelpad=10)
plot4.set_ylabel("Streamflow (MGD)", fontsize = 16,labelpad=10)
plot4.set_xlim(dry_weekly.index.min(),dry_weekly.index.max())
plot4.tick_params(axis='both', labelsize=12)
plot4.xaxis.set_major_formatter(myFmt)

plt.subplots_adjust(hspace=0.5, wspace=0.3)
fig.legend(wet_daily.columns.values, 
           loc='lower center',
           fontsize = 'x-large',
           ncols=2,
           labelspacing = 1,
           bbox_to_anchor=(0.5, -0.01)
           )
plt.show()
